<a href="https://colab.research.google.com/github/Lusilpa/data_science/blob/main/Filmes_Brasileiros.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install openpyxl
!pip install requests
!pip install odfpy

print("! ! ! Instalações concluídas ! ! !")

In [ ]:
import pandas as pd

caminho = '/content/15pb-filmes-pbi-a-pbvi.xlsx'

if not caminho: print("Arquivo não encontrado")

df = pd.read_excel(caminho)

print("Primeiras linhas do arquivo")
display(df.head())

print("\nEstrutura de Dados: ")
df.info()

In [ ]:
colunas = [
    "Título do filme",
    "Direção",
    "Ano",
    "UF"
]

filmes_cols = df[colunas].copy()
filmes_cols

**Filtros Possiveis**

In [ ]:
#Quais as produções por Unidade Federativa?

filtro_uf = df.query("UF == 'AM'")
filtro_uf

In [ ]:
# Quais as produções do ano de 2007?

filtro_ano = df.query("Ano == 2007")
filtro_ano

In [ ]:
# Quais produções foram feitas no Estado de São Paulo no ano de 2007?

filtro_uf_ano = df.query("Ano == 2007 and UF == 'SP'")
filtro_uf_ano

In [ ]:
# Quantas produções foram feitas por Unidade Federativa (UF)?

# Garante que tudo seja texto e remove espaços em branco nas pontas
df['UF'] = df['UF'].astype(str).str.strip()

# Padroniza os separadores
df['UF_limpa'] = df['UF'].str.replace(r'[\s]*/[\s]*', ';', regex=True)
df['UF_limpa'] = df['UF_limpa'].str.replace(r'[\s]*-[\s]*', ';', regex=True)
df['UF_limpa'] = df['UF_limpa'].str.replace(r'[\s]*,[\s]*', ';', regex=True)

# Separa os estados que estavam juntos e joga cada um em uma linha própria
df_explodido = df.assign(UF_limpa=df['UF_limpa'].str.split(';')).explode('UF_limpa')

# Remove espaços extras que sobraram e filtra valores inválidos
df_explodido['UF_limpa'] = df_explodido['UF_limpa'].str.strip().str.upper()
df_final = df_explodido[~df_explodido['UF_limpa'].isin(['NAN', '', 'NONE', 'NULO'])]

# Agora sim: gera a contagem correta e limpa
tabela_limpa = df_final['UF_limpa'].value_counts().reset_index()
tabela_limpa.columns = ['UF', 'N°']

print(tabela_limpa)

In [ ]:
# Quantas produções foram feitas por ano em cada estado?

import matplotlib.pyplot as plt
import seaborn as sns

# Cria a matriz com UF nas linhas (Y) e Ano nas colunas (X)
tabela_matriz = pd.crosstab(index=df_final['UF_limpa'], columns=df_final['Ano'])

print("--- QUANTIDADE DE PRODUÇÕES POR ANO EM CADA ESTADO ---")
display(tabela_matriz.style.background_gradient(cmap='Blues'))